# Activity: Using state-of-the-art generative models

**Initial Due Date: 2026-01-23 10:00AM**  
**Final Due Date: 2026-01-26 4:15PM**

In [ ]:
# Set global constants for notebook (you should not need to modify these values)
SITE_URL = "https://middcs.github.io/csci-1010-w26"

## Learning Objectives

By the end of this notebook, you will be able to:

1.  Use state-of-the-art generative language models via their APIs.
2.  Compute text embeddings using a generative model for similarity comparison and document retrieval.
3.  Build and quantitatively evaluate a genAI-powered application to augment or replace human workflows.

## Part A: Getting started

We will use the [Google Gemini API](https://ai.google.dev/gemini-api/docs) to access state-of-the-art generative models. This was one choice among several available APIs. We chose the Gemini API because it offers a free tier (sort of … more below) that allows us to experiment with these models at no cost and without needing to provide billing information.

To use the API, you will need to create a free API key at <https://aistudio.google.com/app/apikey> via the button at the top right (you should not need to provide any billing information for the free tier). As part of the process you may need to create a project. We called both “cs1010-w26”, but you can use any name.

We will store the API key as a Google Colab secret so that is accessible in your notebooks, but not visible to others (following this [guide](https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb)):

1.  Click on 🔑 Secrets tab in the left panel.

    <img src="https://storage.googleapis.com/generativeai-downloads/images/secrets.jpg" alt="You can find the Secrets tab on the left panel." width=50%>

2.  “Add new secret” with the name `GOOGLE_API_KEY`.

3.  Copy and paste your API key into the `Value` input box of `GOOGLE_API_KEY`.

4.  Toggle the button on the left to allow all notebooks access to the secret.

Get started by authenticating with the Gemini API using your newly acquired API key:

In [ ]:
from google import genai

try:
    # If running in Colab, retrieve the API key from Colab secrets
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
except ImportError:
    # Fallback to retrieving the API key from environment variables
    import os
    GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')

In [ ]:
# Create the API client
client = genai.Client(api_key=GOOGLE_API_KEY)

Print out the available models:

In [ ]:
for model in client.models.list():
    print(model.name)

There are many models available, including specific versions, models optimized for chat, models optimized for embeddings, etc. Unfortunately, not all models are available on the free tier. And some models that are available, such as the general purpose “gemini-2.5” series, have very limited free tier usage quotas (e.g., 20 requests per day!). For this reason, we will default to the smaller “gemma” models for this activity, which have more generous free tier quotas.

Select the model you want to use for general content generation:

In [ ]:
MODEL_ID = "gemma-3-27b-it" # @param ["gemma-3-27b-it", "gemini-2.5-flash"]

You should now be able to generate content using the select model:

``` python
from IPython.display import Markdown

response = client.models.generate_content(
    model=MODEL_ID,
    contents="See you",
)

# Use `display(Markdown(response.text))` to render the Markdown in your notebook. We 
# don't do so by default so the output can be distinguished from the notebook text.
response.text
```

The output is more sophisticated than what we saw with GPT2! Unfortunately the Gemini API does not provide access to the model’s internal logits, so we do not have the same visibility into how the model is making its predictions as we did with GPT2. But we can still experiment with different decoding strategies by specifically setting the generation parameters (e.g., temperature, max output tokens, etc.) and by providing different (system) prompts.

Try the following prompt with the Gemma model. Here we instruct the model to behave like a raw language model that just predicts the next token without any self-correction or advanced reasoning. Using the configuration parameters, we reduce the temperature to 0.0 to make the model more deterministic, and limit the output length to 5 tokens (similar to the output we generated with GPT-2). The temperature parameter controls the randomness of the model’s output; a temperature of 0.0 makes the model deterministic, while higher values introduce more randomness.

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents="You are a raw language model. Respond to prompts as if you are predicting the next most likely words, without self-correction, opinions, or advanced reasoning. Keep responses short and focused purely on text continuation.\n See you",
    config=genai.types.GenerateContentConfig(
        maxOutputTokens=5,
        temperature=0.0,
    )
)
response.text

As an example, a corresponding prompt for the more sophisticated Gemini family of models is shown below. This is just for reference, since the number of allowed requests for those models are so limited. With Gemini-family models we can provide an explicit “system” prompt via the `system_instruction` field in the generation configuration. As described in the [quickstart guide](https://github.com/google-gemini/cookbook/blob/main/quickstarts/System_instructions.ipynb):

> System instructions allow you to steer the behavior of the model. By setting the system instruction, you are giving the model additional context to understand the task, provide more customized responses, and adhere to guidelines over the user interaction. Product-level behavior can be specified here, separate from prompts provided by end users.

Further we can turn off the “thinking” capabilities of the model by setting the `thinking_budget` to 0. This disables the model’s ability to perform multi-step reasoning or self-reflection, making it behave more like a language model that simply predicts the next token based on the input prompt.

``` python

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="See you",
    config=genai.types.GenerateContentConfig(
        system_instruction="You are a raw language model. Respond to prompts as if you are predicting the next most likely words, without self-correction, opinions, or advanced reasoning. Keep responses short and focused purely on text continuation.",
        maxOutputTokens=5,
        temperature=0.0,
        candidateCount=5,
        thinking_config=genai.types.ThinkingConfig( # Disable "thinking" for this raw prediction task
            thinking_budget=0
        ),
    )
)

# Concatenate all of the candidate outputs.
response.text
```

## Part B: Working with embeddings

So far we have used generative models for text generation, but many models (including Gemini) also support generating embeddings. The models’ generative capabilities are enabled by semantic understanding of the input prompts. That semantic understanding is captured in the model’s internal representations, which can be extracted as *embeddings*. Embeddings are high-dimensional vector representations of text that capture semantic meaning, and can be used for tasks like similarity comparison, retrieval and more.<span class="column-margin margin-aside">For example, these embeddings are often used as the feature inputs to separate machine learning models for classifying text.</span> Our expectation is that semantically similar texts will have similar embeddings, i.e., be “close together” in the high-dimensional embedding space. These embeddings are often generated from the hidden states of the model, i.e., “upstream” of final token prediction layers we worked with previously.

The Gemini API provides dedicated embedding models and [dedicated methods for generating embeddings](https://ai.google.dev/gemini-api/docs/embeddings).

As a motivating application, let’s explore Middlebury’s Computer Science (CS) course offerings with the goals of understanding how similar courses different courses might be and identifying relevant courses based on our interests (as expressed in natural language queries). We will use embeddings to do both. <span class="column-margin margin-aside">This part of the activity was adapted from a [Google Gemini Cookbook example](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Embeddings.ipynb)</span>.

Start with our standard data science imports:

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

### Exercise B1: Obtaining the course listings

To start, we will need to get the course listings from Middlebury’s CS course catalog page: <https://www.middlebury.edu/college/academics/computer-science/courses>. This is unfortunately trickier than we might hope, since that page does not provide an API or a data download option. Instead, we will need to scrape the course listings from the HTML of the page.

> #### ❗ Considerations when web scraping
>
> Here we are scraping data from a Middlebury web page for internal educational purposes. Further, Middlebury’s [`robot.txt` file](https://en.wikipedia.org/wiki/Robots.txt) specifically allows bot access to the page of interest. However, before scraping data from any web page, you should familiarize yourself with your country’s laws and the general ethical principles regarding web scraping. Always check the site’s `robots.txt` file (by appending `/robots.txt` to the base URL) to see if scraping is allowed. Additionally, review the website’s terms of service to ensure compliance with their policies regarding data usage and scraping.

The Gemma family of models does not support direct web access, but the more advanced Gemini family of models does. It would be ideal if we could use a Gemini model to directly scrape the course listings for us. However, when I try to the following prompt I run into a RECITATION error indicating that “Token generation stopped because the content potentially contains copyright violations.”

``` python
# This prompt may produce a RECITATION error, i.e., model won't reproduce online data
# Python note: The triple quotes create a multi-line string
prompt = """
  Create a listing the course numbers, names an descriptions from all the courses available at https://www.middlebury.edu/college/academics/computer-science/courses. Use this JSON schema:

  Course = {'number': str, 'short_name': str, 'long_name': str, 'description': str }
  Return: list[Course]

  For example:
  [
    {
      "number": "CSCI 0105",
      "short_name": "Algorithmic World",
      "long_name": "Understanding Our Algorithmic World",
      "description": "n this course through lectures, labs, and discussions, we will examine the nature of computers and their role in our lives...."
    },
    ...
  ]
"""

config = {
   "tools": [{"url_context": {}}],
}

raw_response = client.models.generate_content(
    contents=prompt, model="gemini-2.5-flash", config=config
)
```

Instead we will use a Gemma model to generate Python code that performs the web scraping for us. We can then run the generated code to obtain the course listings. Users familiar with agentic models may have success with the model learning the page structure on its own, but we did not. Instead we will create a more detailed prompt that describes the HTML structure of the page to help the model generate the correct code. Using “View source”, we extract a relevant snippet of HTML for a single course listing to include in the prompt.

> ### 💡 Interactive/exploratory prompt development for Gemini models
>
> For an iterative task like this you might find it helpful to use the online chat interface for Gemini models at <https://aistudio.google.com>. That interface allows you to quickly try out different prompts and see the model’s responses. It also provides access to the more sophisticated Gemini models, and their URL context, code execution and other capabilities.

In [ ]:
prompt = """
Write Python code to scrape the course information from the URL https://www.middlebury.edu/college/academics/computer-science/courses. The code should extract the course number, short name, long name, and description for each course. The output should be a list of Python dictionaries, where each dictionary represents a course and has the following keys: 'number', 'short_name', 'long_name', and 'description'.

Each course is contained withing a `<div>` with the following HTML structure:

```html
<div class="accordion-item js-accordion-item-15419-CSCI0105 has-toggler is-toggled">
  <a href="#midd-accordion-item-label-15419-CSCI0105" class="accordion-item__link has-toggler is-toggled" data-toggle-target=".js-accordion-item-15419-CSCI0105" aria-expanded="true" aria-controls="midd-accordion-content-15419-CSCI0105" aria-labelledby="midd-accordion-item-label-15419-CSCI0105" role="tab">
    <h3 class="accordion-item__title" id="accordion-item-label-15419-CSCI0105">

      <span id="midd-accordion-item-label-15419-CSCI0105">
        <p class="accordion-item__label--bold">
          CSCI 0105
                            </p>
        Algorithmic World
      </span>

      <svg class="icon accordion-item__icon " focusable="false" aria-hidden="true">
        <use xlink:href="#icon-chevron-down"></use>
      </svg>

    </h3>
  </a>
  <div class="accordion-item__content--column" id="midd-accordion-content-15419-CSCI0105" aria-labelledby="midd-accordion-item-label-15419-CSCI0105" role="tabpanel">
    <p class="accordion-item__sub-title">Course Description</p>
    <div class="typography">
      <p><strong>Understanding Our Algorithmic World</strong><br> In this course through lectures, labs, and discussions, we will examine the nature of computers and their role in our lives. We will use the lens of multimedia programming to learn basic computer programming and how computers represent and manipulate many common forms of data, such as text and images. We will also talk about the history of computers and learn how they interoperate to create the world we know today, and we will examine the societal impacts of technology on our lives, including implications for privacy, access to resources, and the increasing role of algorithms in shaping our world. (not open to students who have taken CSCI 0145 or higher) 3 hrs. lect./lab</p>
    </div>

          <p class="accordion-item__sub-title">Terms Taught</p>
      <div class="typography">
                  Spring 2022,                  Fall 2024              </div>
    
          <p class="accordion-item__sub-title">Requirements</p>
      <div class="typography">
                  <strong>DED</strong>              </div>
    
    <p class="accordion-item__sub-title"><a href="https://catalog.middlebury.edu/courses/view/course-CSCI0105" title="View Algorithmic World in the Course Catalog">View in Course Catalog</a></p>
  </div>
</div>
```

The corresponding output should look like:

```python
{'number': 'CSCI 0105',
  'short_name': 'Algorithmic World',
  'long_name': 'Understanding Our Algorithmic World',
  'description': 'In this course through lectures, labs, and discussions, we will examine the nature of computers and their role in our lives. ...'
}
```
Your code should handle multiple courses and correctly parse their respective details from the HTML structure of the page. You may use libraries like requests for fetching the page content and BeautifulSoup for parsing HTML.
"""

response = client.models.generate_content(
    contents=[prompt], model=MODEL_ID,
)
display(Markdown(response.text))

Refine the prompt above and or the returned code to create a function named `get_middlebury_courses` that takes a URL as its argument and returns a list of course listings as dictionaries. Here is a template for a solution. The generated code will likely (certainly) need refinement to work correctly. To enable you to make progress on the rest of the activity, even if the scraping code is not working, a pre-scraped version of the course listings is provided below.

In [ ]:
import requests
from bs4 import BeautifulSoup

def get_middlebury_courses(url: str) -> list[dict]:
    # Code initially generated by gemma-3-27b and then extensively refined manually
    response = requests.get(url)
    response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)

    soup = BeautifulSoup(response.content, 'html.parser')

    course_listings = []
    course_divs = soup.find_all('div', class_='accordion-item')

    for div in course_divs:
        # TODO: Your code here
    return course_listings   

course_listings = get_middlebury_courses("https://www.middlebury.edu/college/academics/computer-science/courses")

The resulting list of dictionaries can be directly read into a Pandas DataFrame for analysis. Don’t change the name `courses` for this DataFrame (or any other specified variables), as it is used in the subsequent steps.

In [ ]:
# Convert the list of course dictionaries into a DataFrame
courses = pd.DataFrame(course_listings)
courses.head()

# If you want to load the pre-scraped data instead, comment out the two lines above 
# and uncomment the following lines:
# url = f"{SITE_URL}/data/midd_cs_courses.csv"
# courses = pd.read_csv(url)

### Exercise B2: Generating embeddings

With the text, we can now generate the actual embeddings. To do so we will use an embedding model and the `embed_content` method. [By default](https://ai.google.dev/gemini-api/docs/embeddings#control-embedding-size) the Gemini API generates 3072-dimensional embeddings. To make things more manageable we will request smaller (but still large!) 768-dimensional embeddings instead.<span class="column-margin margin-aside">The smallest of the recommend sizes</span> Reviewing the documentation we see a note about [ensuring embedding quality](https://ai.google.dev/gemini-api/docs/embeddings#quality-for-smaller-dimensions) by normalizing embeddings smaller than 3072. We will incorporate that normalization step and the example code into our analysis.

In [ ]:
EMBEDDING_MODEL_ID = "gemini-embedding-001" # @param ["gemini-embedding-001"]

In [ ]:
# Extract a single course row for testing
course = courses.loc[0]

# response.embeddings[0].values is a list of floats containing the embedding
response = client.models.embed_content(
    model=EMBEDDING_MODEL_ID,
    contents=course['description'],
    config=genai.types.EmbedContentConfig(output_dimensionality=768)
)

# Perform normalization as suggested in the documentation
embedding = np.array(response.embeddings[0].values)
norm_embedding = embedding / np.linalg.norm(embedding)
norm_embedding

While we can repeat the above request for each course, it is more efficient to generate all the embeddings in a single request by passing a list of contents. We can then easily normalize all embeddings in a single step as well.

In [ ]:
response = client.models.embed_content(
    model=EMBEDDING_MODEL_ID,
    contents=courses['description'].to_list(),
    config=genai.types.EmbedContentConfig(output_dimensionality=768)
)

The resulting `response` contains all of the embeddings in the list `response.embeddings`. Complete the code below to construct a 2-D NumPy array `norm_embeddings` (44x768) where each row corresponds to a normalized embedding for a course description. Recall that you can specify the axis along which to compute reductions like `np.linalg.norm`.

The division is element-wise, but your two operands do not have the same shape. In some cases, like for the single embedding above, the necessary broadcasting is unambiguous and will be performed automatically. If not, or if you want to be explicit, you can specify the desired broadcast dimension(s) with square brackets and `np.newaxis` to indicate the broadcasted axes. For example for a 1-D array, `array[:, np.newaxis]` would create a 2-D array by broadcasting across the columns (i.e., an array where all the elements in each row are the same).

In [ ]:
# TODO: Your code here

### Exercise B3

You should have created the 2-D array, `norm_embeddings` above. We are now ready to compute similarities between the course descriptions. We will use the Cosine Similarity metric, which is the dot product of the normalized embeddings.<span class="column-margin margin-aside">In our case since embeddings are already normalized, it is the same as the dot product.</span> Higher values mean more similar, i.e., a value of 1.0 means identical embeddings, while a value of 0.0 means orthogonal (dissimilar) embeddings.

The `sklearn` library provides a function to compute the pairwise cosine similarities (and other distance metrics). Complete the code below to compute the pairwise cosine similarities between all course descriptions and store the result in a Pandas DataFrame named `similarities`, where both the rows and columns are indexed by the course numbers. Notice that values along the diagonal are (should be) 1.0 since they represent the similarity of each course with itself.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
# TODO: Your code here
similarities

We can visualize the similarity matrix as a heatmap using the `seaborn` library (check out [sns.heatmap](https://seaborn.pydata.org/generated/seaborn.heatmap.html#seaborn-heatmap)). Complete the code below to create the heatmap. Since we have many courses we will want to ensure all the tick labels are included (with the `xticklabels` and `yticklabels` parameters set to `True`) and turn off the values inside the heatmap cells (with `annot=False`). Finally, we will set a smaller tick label font size for readability.

In [ ]:
plt.figure(figsize=(8, 6))
# TODO: Your code here
ax.tick_params(axis='both', which='major', labelsize=8)
plt.show()

What do you observe about the similarity results? Add a new text cell immediately below this paragraph (or edit the placeholder text) to briefly describe two aspects of the data you would/did examine to build confidence that the embeddings are capturing meaningful semantic relationships between the courses.

*[TODO: Your response here]*

### Exercise B4

There are many applications for embeddings, including document retrieval. Here we will use the course description embeddings to identify relevant courses based on a natural language query. Our hypothesis is that the query embedding will be similar to the embeddings of relevant courses (course descriptions). Thus selecting the most similar course embedding to the query embedding should return relevant courses.

The Gemini API provides embeddings optimized for different tasks, including document retrieval. The code below uses the `"RETRIEVAL_DOCUMENT"` type for the course descriptions and then `"RETRIEVAL_QUERY"` for the query itself. Complete the code below by copying your normalization code from above to compute `norm_doc_embeddings`. These will serve as the “vector database” that we will query against.

In [ ]:
response = client.models.embed_content(
    model=EMBEDDING_MODEL_ID,
    contents=courses['description'].to_list(),
    config=genai.types.EmbedContentConfig(
        task_type="RETRIEVAL_DOCUMENT", 
        output_dimensionality=768
    )
)

# TODO: Your code here

We now generate an embedding for a natural language query. Our hypothesis is that the query embedding will be similar to the embeddings of relevant course descriptions.

In [ ]:
query = "I am interested in machine learning and AI"

response = client.models.embed_content(
    model=EMBEDDING_MODEL_ID,
    contents=query,
    config=genai.types.EmbedContentConfig(
        task_type="RETRIEVAL_QUERY", 
        output_dimensionality=768
    )
)

query_embedding = np.array(response.embeddings[0].values)
norm_query_embedding = query_embedding / np.linalg.norm(query_embedding)

Complete the function `similar_courses` below to return the `k` rows in the `courses` data frame most relevant (most similar) to our query. We will use the dot product as the similarity metric (check out [`np.dot`](https://numpy.org/doc/stable/reference/generated/numpy.dot.html)). A helper function `topk` is provided to return the sorted indices of the top `k` largest values in an 1-D array. Recall the DataFrame `iloc` attribute enables selecting DataFrame rows by integer index.

In [ ]:
def topk(array: np.ndarray, k: int) -> np.ndarray:
    """Return the sorted indices of the top-k largest elements in the array."""
    topk_unsorted = np.argpartition(array, -k)[-k:]
    return topk_unsorted[np.argsort(array[topk_unsorted])][::-1]

def similar_courses(courses: pd.DataFrame, doc_embeddings: np.ndarray, query_embedding: np.ndarray, k=5) -> pd.DataFrame:
    """Return the top k most similar courses to the query."""
    # TODO: Your code here

In [ ]:
similar_courses(courses, norm_doc_embeddings, norm_query_embedding, k=5)

Are those courses relevant to the query? Hopefully, yes! Try other queries of interest to you and see what courses are returned. Add a new text cell immediately below this paragraph (or edit the placeholder text) to briefly describe at least one other query you tried and the results you obtained.

*[TODO: Your response here]*

## Part C: Autonomous workflow with generative models

One of the potential applications for generative models is to augment or replace labor-intensive human workflows that might be hard to automate with traditional programming techniques (e.g., the rules rely on context). Here we will prototype a workflow that evaluates receipts submitted as part of an expense report, e.g., for employee travel or student club expenses. The goal is to determine if the receipt is valid (i.e., meets the reimbursement policy). To do so we will need to extract relevant information (e.g., total amount, date, vendor, etc.) from the receipt, typically an image, and evaluate those items in context. We will use a modern generative model, with its multi-modal capabilities (i.e., handle text and image inputs) to perform these tasks.

This activity was adapted from an [OpenAI cookbook](https://cookbook.openai.com/examples/partners/eval_driven_system_design/receipt_inspection). As the authors note, this prototype is not intended to compete with existing solutions in this space, but rather to illustrate some the capabilities and considerations when working with generative models for workflow automation. Nor is it intended to exactly follow Middlebury’s reimbursement policies.

The general workflow:

1.  Users, say student clubs, submit receipts as images as part of an expense report along with the intended purpose of the expense.
2.  The finance team reviews the receipt to approve or further audit the expense report.

For our prototype, let’s break the problem into two separate tasks (prompts) that implement the steps above.

### Step 1: Extract relevant information from the receipt image

Following the upstream example, we will use images from the [CC BY 4.0-licensed](https://creativecommons.org/licenses/by/4.0/) [Receipt Handwriting Detection Dataset](https://universe.roboflow.com/newreceipts/receipt-handwriting-detection), a collection of scanned receipts. Although that dataset was not created for this purposes, these images are useful example inputs for our prototype.

<figure>
<img src="https://middcs.github.io/csci-1010-w26/data/20230722_181229_Raven_Scan_3_jpeg.rf.bd604286f5708cc5e42f7863e7723290.jpg" alt="Example receipt sourced from the Receipt Handwriting Detection Dataset" />
<figcaption aria-hidden="true">Example receipt sourced from the <a href="https://universe.roboflow.com/newreceipts/receipt-handwriting-detection">Receipt Handwriting Detection Dataset</a></figcaption>
</figure>

Following the [documentation for working with images in the Gemini API](https://ai.google.dev/gemini-api/docs/image-understanding), we will upload the image file and then provide both the image and a prompt to the model to extract relevant information from the receipt. The prompt below is adapted for Gemma from the OpenAI cookbook example linked above. Note that unlike the Gemini family of models, the Gemma models do not support specifying the output structure in the request configuration, so we include that in the prompt itself. We also need to do additional work to parse the model’s response into structured data.

> #### ❓ Structured data and JSON
>
> You can configure or instruct modern generative models to return data that follows a specific JSON structure, often termed [“structured output”](https://ai.google.dev/gemini-api/docs/structured-output). JSON (JavaScript Object Notation) is a lightweight data-interchange format for communicating between computer systems, that is also easy(?) for humans to read and write. It is commonly used for transmitting data in web applications. Generating structured output enables us to work with the model response programmatically. In this as in other aspects, the model may not always produce the desired output structure. Additional validation and error handling may be needed. As you read the documentation and other online resources, you may see references to “schema” or “schema validation”. A schema defines the expected structure and data types of the JSON output. Schema validation is the process of checking if the generated JSON adheres to the defined schema. [Pydantic](https://docs.pydantic.dev/latest/) is a popular Python library for data validation and settings management using Python type annotations. It can be used to define schemas and validate JSON data against those schemas. Many models also directly accept Pydantic models as part of the request configuration for structured output.

In [ ]:
from typing import Any

def extract_json(response_text: str) -> Any:
    """Extract the JSON portion from the model response text."""
    import json

    # Remove JSON markdown formatting if present (i.e., ```json ... ```)
    response_text = response_text.removeprefix("```json").removesuffix("```")
    return json.loads(response_text)

In [ ]:
extract_prompt = """
Given an image of a retail receipt, extract all relevant information and return is as JSON.

# Task Description

Carefully examine the receipt image and identify the following key information:

1. Merchant name and any relevant store identification
2. Location information (city, state, ZIP code)
3. Date and time of purchase
4. All purchased items with their:
   * Item description/name
   * Item code/SKU (if present)
   * Category (infer from context if not explicit)
   * Regular price per item (if available)
   * Sale price per item (if discounted)
   * Quantity purchased
   * Total price for the line item
5. Financial summary:
   * Subtotal before tax
   * Tax amount
   * Final total
6. Any handwritten notes or annotations on the receipt (list each separately)

## Required format:

{
  "merchant": str | null,
  "location": {
    "city": str | null,
    "state": str | null,
    "zip_code": str | null
  },
  "time": str | null,
  "items": [
    {
      "description": str | null,
      "item_code": str | null,
      "category": str | null,
      "regular_price": str | null,
      "sale_price": str | null,
      "quantity": str | null,
      "total_price": str | null
    },
    ...
  ],
  "subtotal": str | null,
  "tax": str | null,
  "total": str | null,
  "handwritten_notes": [str, ...]
}

## Important Guidelines

* If information is unclear or missing, return null for that field
* Format dates as ISO format (YYYY-MM-DDTHH:MM:SS)
* Format all monetary values as decimal numbers
* Distinguish between printed text and handwritten notes
* Be precise with amounts and totals
* For ambiguous items, use your best judgment based on context

Your response should be JSON and complete, capturing all available information from the receipt.
"""

image = client.files.upload(file=f"{SITE_URL}/data/20230722_181229_Raven_Scan_3_jpeg.rf.bd604286f5708cc5e42f7863e7723290.jpg")

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[image, extract_prompt],
)

# Gemma returns JSON as markdown formatted text, i.e. wrapped in ```json ... ```, so
# use the helper function to strip that out before parsing into Python types.
receipt_data = extract_json(response.text)
receipt_data

That is a very promising start! Each run may produce slightly different results, but likely the model does a good job overall of extracting the relevant information from the receipt image.

### Step 2: Evaluate the receipt data against reimbursement policies

In [ ]:
import json

# Create a template string for use with the `format` method. We will replace the `{purpose}` and
# `{receipt_data}` placeholders for the actual request. To include literal curly braces in the
# template, we need to double them as `{` and `}`.
audit_prompt = """
Evaluate this receipt data to determine if it need to be audited based on the following criteria:

1. INCONSISTENT_EXPENSE: The items are NOT consistent with the stated purpose of the expense.
   - For example, if the purpose is "travel" but the receipt is for office supplies, set this to TRUE.
   - If the purpose is "travel" and the receipt is for gas/fuel, this would be FALSE because gas IS travel-related.
   - Other examples for TRUE would be durable items like electronics, furniture, for meal expenses.
   - Always set this to TRUE for alcoholic beverages regardless of purpose.

2. AMOUNT_OVER_LIMIT: The total amount exceeds $50

3. HANDWRITTEN_X: There is an "X" in the handwritten notes

For each criterion, determine if it is violated (true) or not (false). Provide your reasoning for each decision, and make a final determination on whether the receipt needs auditing. A receipt needs auditing if ANY of the criteria are violated.

Return a JSON response with the following required format:

{{
  "INCONSISTENT_EXPENSE": bool,
  "AMOUNT_OVER_LIMIT": bool,
  "HANDWRITTEN_X": bool,
  "reasoning": str,
  "needs_audit": bool
}}
  
# Purpose of expense:
{purpose}

# Receipt data:
{receipt_data}
"""

response = client.models.generate_content(
    model=MODEL_ID,
    # Construct the complete prompt by filling in the placeholders via keyword arguments
    contents=audit_prompt.format(
        purpose="Club halloween party supplies",
        receipt_data=json.dumps(receipt_data, indent=2),
    )
)

audit_data = extract_json(response.text)
audit_data

Again, that is likely a very promising start! Each run will likely produce slightly different results, but overall the model does a good job of extracting the relevant information from the receipt image.

### Exercise C1

Make sure you have run the code above to complete examples of steps 1 and 2 and populate the variables `receipt_data` and `audit_data`. The examples above demonstrate the two key steps in our prototype workflow. But, we did not address evaluation. Add a new text cell immediately below this paragraph (or edit the placeholder text) to briefly outline how you would evaluate the performance of this prototype system. A satisfactory answer will describe an evaluation dataset, propose as least one “end-to-end” metric, and discuss how you might evaluate the individual steps.

*[TODO: Your response here]*

### Exercise C2

Some aspects of the workflow we can evaluate programmatically, e.g., whether the extracted receipt data is mathematically consistent. Write a function named `check_receipt_math` that takes the extracted receipt data (the output of step 1) as its argument and returns `True` if the line item totals sum to the subtotal, and subtotal plus tax equals the total, else return `False`. You can assume that all monetary values are strings representing decimal numbers (e.g., `"12.34"`) and that you only need to consider each item’s `"total_price"`. The built-in `float` function can convert strings to floating point numbers for arithmetic operations.

In [ ]:
# Recall comparing floats for equality can be tricky due to precision issues. In a production setting we
# might consider using the `decimal` module for exact decimal arithmetic. For simplicity, here we will use
# floats. You can use `numpy.isclose` to compare floats for "closeness" rather than exact equality, e.g.,
# `atol=0.01` to check the values are within 1 cent.

def check_receipt_math(receipt_data: dict) -> bool:
    """Return True if the item totals sum to subtotal, and subtotal plus tax equals total."""
    # TODO: Your code here
    return True

Test your function on the receipt data extracted above and some example inputs with both and incorrect math.

In [ ]:
print("Receipt math correct:", check_receipt_math(receipt_data))

subtotal_differs = {
    "items": [ {"total_price": "10.00"}, {"total_price": "15.10"}],
    "subtotal": "25.00",
    "tax": "2.51",
    "total": "27.51"
}
print("Subtotal differs returns False:", check_receipt_math(subtotal_differs))

tax_differs = {
    "items": [ {"total_price": "10.00"}, {"total_price": "15.00"}],
    "subtotal": "25.00",
    "tax": "3.00",
    "total": "27.51"
}
print("Tax differs returns False:", check_receipt_math(tax_differs))

valid_receipt = {
    "items": [ {"total_price": "10.00"}, {"total_price": "15.00"}],
    "subtotal": "25.00",
    "tax": "2.50",
    "total": "27.50"
}
print("Valid receipt returns True:", check_receipt_math(valid_receipt))

### Exercise C3

Add a new text cell immediately below this paragraph (or edit the placeholder text) to briefly hypothesize possible situations where this automated workflow might produce incorrect results in the student club context. Could those failure modes disproportionately affect certain groups or individuals? A satisfactory answer will describe at least one possible failure mode and discuss its potential for differential impact.

*[TODO: Your response here]*

## Collaboration statement

In a new text cell immediately below this paragraph (or by editing this text cell to add a paragraph), briefly list who or what you collaborated with and how. Cite any sources here or with relevant inline comments in your code. Acknowledge all contributors, both people and AI, and what portions of this notebook they contributed. You do not need to cite or acknowledge any material provided in the starter file(s).

## Submitting your notebook

You will simultaneously submit the following two files to the relevant assignment on [Gradescope](https://gradescope.com) via the “Upload option” (guide [here](https://guides.gradescope.com/hc/en-us/articles/21865616724749-Submitting-a-Code-assignment)). **Both files must be uploaded at the same time and the file names must match the specification exactly for the autotesting to run successfully.**

1.  `activity_using_generative_models.ipynb`: Your completed IPython notebook. You can obtain this via the “File→Download→Download .ipynb” menu option in Colab.
2.  `activity_using_generative_models.py`: Your completed IPython notebook as a Python file. You can obtain this via the “File→Download→Download .py” menu option in Colab. This file is used to provide line-level feedback on your submission.

You can submit multiple times, with only the most recent submission (before the final due date) assessed for credit. Gradescope will run a series of automated unit tests on your notebook (which may takes 10s of seconds depending on the complexity of the notebook). Note that the tests performed by Gradescope are limited. Passing all of the visible tests does not guarantee that your submission correctly satisfies all of the requirements of the assignment.